In [ ]:
# Cylinder control
#
# Plain cylinders of every diameter in CYLINDER_DIAMETERS (config.py), on the
# same animals and insertion sites as the electrode comparison. Separates the
# effect of thickness alone from the effect of probe shape.
# Outputs: results/diameters/.
import os
import pandas as pd

from config import (ANIMALS, CYLINDER_DIAMETERS, RESULTS_DIR, SEG_CACHE_DIR, tiff_path,
                    Y_CENTER, POS_NUM, DEPTH_LIMIT)
from Volume_bleeding import load_volume, process_cone_positions
from Number_bleeding import load_segmentation, process_cone_positions_num

out_dir = os.path.join(RESULTS_DIR, "diameters")
os.makedirs(out_dir, exist_ok=True)

In [ ]:
# For each animal: load it once, then run both metrics for every diameter.
# Writes <animal>_Volume.csv and <animal>_Number.csv: one row per insertion
# site, one column per diameter.

for animal in ANIMALS:
    path = tiff_path(animal)
    name = os.path.splitext(os.path.basename(path))[0]
    print(f"=== {name} ===")
    img = load_volume(path)
    seg = load_segmentation(path, SEG_CACHE_DIR, img)

    volume, number = {}, {}
    for d in CYLINDER_DIAMETERS:
        print(f"  diameter {d} um")
        # A cylinder: one shank spanning the full depth at constant diameter, no tip.
        geom = dict(shank_length=DEPTH_LIMIT, shank_base_diameter=d, shank_top_diameter=d,
                    tip_length=0, tip_base_diameter=d, tip_top_diameter=d,
                    depth_limit=DEPTH_LIMIT, pos_num=POS_NUM)
        volume[f"{d}um"] = process_cone_positions(img, Y_CENTER, **geom)
        number[f"{d}um"] = process_cone_positions_num(seg, Y_CENTER, **geom)

    for metric, table in (("Volume", volume), ("Number", number)):
        df = pd.DataFrame(table)
        df.insert(0, "position", range(POS_NUM))
        df.to_csv(os.path.join(out_dir, f"{name}_{metric}.csv"), index=False)
print(f"Done. CSVs in {out_dir}")